In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression


Der direkte Vergleich: Pandas vs. Sklearn Pipeline


Schritt	In Pandas (Vorher händisch)	Mit Sklearn Pipeline (Automatisiert)
Fehlende Werte	df.fillna() ändert den aktuellen Datensatz.	Ein SimpleImputer merkt sich den Trainings-Mittelwert für die Zukunft.
Skalierung	Direktes Ändern der Spalten im DataFrame.	StandardScaler wird als fester Baustein in die Kette integriert.
Kreuzvalidierung (GridSearch)	Extrem kompliziert, da man in jeder Runde neu in Pandas trennen müsste.	Kinderleicht. Die Pipeline verhindert automatisch, dass Daten zwischen den Runden vermischt werden.
Produktivbetrieb (Deployment)	Du musst den Pandas-Code als Skript mitliefern.	Du speicherst ein einziges Pipeline-Objekt ab und lädst es auf dem Server.


In [2]:
# 1. Beispiel-Daten erstellen (mit einer fehlenden Zahl / NaN)
data = {
    'Quadratmeter': [50, 70, np.nan, 100, 120],
    'Preis': [150000, 220000, 250000, 310000, 380000]
}
df = pd.DataFrame(data)

In [3]:
df

,Quadratmeter,Preis
0,50.0,150000
1,70.0,220000
2,NaN,250000
3,100.0,310000
4,120.0,380000


In [4]:
# Features (X) und Zielvariable (y) trennen
X = df[['Quadratmeter']]
y = df['Preis']


In [6]:
type(X)

pandas.DataFrame

In [7]:
# In Trainings- und Testdaten aufteilen
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [12]:
X_train

,Quadratmeter
4,120.0
2,NaN
0,50.0
3,100.0


In [13]:
X_test  

,Quadratmeter
1,70.0


In [8]:
# =====================================================================
# 2. DIE PIPELINE DEFINIEREN
# =====================================================================
# Hier verknüpfen wir die Bereinigung, Skalierung und das Modell
pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),  # 1. Fehlende Werte mit dem Median auffüllen
    ('scaler', StandardScaler()),                  # 2. Daten normalisieren (Mittelwert=0, Varianz=1)
    ('regressor', LinearRegression())              # 3. Die lineare Regression
])

In [9]:
# =====================================================================
# 3. PIPELINE TRAINIEREN AND ANWENDEN
# =====================================================================

# Trainieren: Der Imputer und Scaler lernen NUR von X_train!
pipeline.fit(X_train, y_train)

# Vorhersage für die Testdaten machen
# X_test wird jetzt AUTOMATISCH im Hintergrund bereinigt und skaliert
predictions = pipeline.predict(X_test)

print("Vorhersage für die Testdaten:", predictions)

Vorhersage für die Testdaten: [202897.19626168]


Wenn man pipeline.predict(neue_kunden) aufruft, passieren drei Schritte vollautomatisch nacheinander, ohne dass dein originaler Datensatz verändert wird:
1. imputer: Die fehlende Zahl (NaN) beim zweiten neuen Kunden wird automatisch durch den Median ersetzt, den die Pipeline damals aus den Trainingsdaten gelernt hat.
2. scaler: Die Quadratmeterzahlen werden exakt mit dem Mittelwert und der Abweichung der Trainingsdaten normalisiert.
3. regressor: Das fertige Lineare Regressionsmodell berechnet den finalen Preis

In [10]:
# =====================================================================
# 4. NEUE DATEN IN DER ZUKUNFT (Der eigentliche Vorteil!)
# =====================================================================
# Morgen kommen komplett neue Kundendaten rein – wieder mit einer Lücke (NaN)!
neue_kunden = pd.DataFrame({'Quadratmeter': [85, np.nan]})

# Kein Pandas-Code nötig! Einfach predict aufrufen:
neue_vorhersagen = pipeline.predict(neue_kunden)
print("Vorhersage für neue Kunden:", neue_vorhersagen)

Vorhersage für neue Kunden: [249299.06542056 295700.93457944]


In [11]:
neue_kunden

,Quadratmeter
0,85.0
1,NaN
